# Phase P7: Experimental Analysis, Figures & Report Generation
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thinkTinkerGreen/train_with_reversibility/blob/main/notebooks/06_analysis.ipynb)


In [ ]:
import os, sys, json, subprocess
SMOKE = os.environ.get("REVLLM_SMOKE", "0") == "1"
IN_COLAB = "google.colab" in sys.modules

REPO_URL = "https://github.com/thinkTinkerGreen/train_with_reversibility.git"
REPO_REF = "main"

if IN_COLAB and not SMOKE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/revllm/data', exist_ok=True)
    os.makedirs('/content/drive/MyDrive/revllm/runs', exist_ok=True)
    os.makedirs('/content/drive/MyDrive/revllm/probes', exist_ok=True)

    repo_dir = "/content/train_with_reversibility"
    if not os.path.exists(repo_dir):
        print(f"Cloning {REPO_URL} into {repo_dir}...")
        subprocess.run(["git", "clone", REPO_URL, repo_dir], check=True)
    else:
        print(f"Repository already present at {repo_dir}. Fetching updates...")
        subprocess.run(["git", "-C", repo_dir, "fetch", "--all"], check=False)
        subprocess.run(["git", "-C", repo_dir, "reset", "--hard", f"origin/{REPO_REF}"], check=False)

    os.chdir(repo_dir)
    deps = ["hatchling", "pyyaml", "tokenizers", "huggingface-hub", "datasets", "tqdm"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + deps, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)
else:
    curr = os.path.abspath(os.getcwd())
    repo_dir = os.path.dirname(curr) if os.path.basename(curr) == "notebooks" else curr
    os.chdir(repo_dir)

for mod in list(sys.modules.keys()):
    if mod.startswith('scripts') or mod.startswith('revllm'):
        del sys.modules[mod]

if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)
src_dir = os.path.join(repo_dir, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print(f"Current working directory: {os.getcwd()}")
print(f"Execution mode: {'Smoke (CPU/CI)' if SMOKE else 'Colab GPU'}")


In [ ]:
# Sync all results from Google Drive to local repo
if IN_COLAB and not SMOKE:
    import shutil
    drive_root = "/content/drive/MyDrive/revllm"
    if os.path.exists(drive_root):
        print(f"Syncing results from {drive_root}...")
        for src_rel in ["env.json", "preflight.json", "probes/batch_probe.json", "runs/variant_selection.json"]:
            full_src = os.path.join(drive_root, src_rel)
            if os.path.exists(full_src):
                dest = os.path.join("results", os.path.basename(src_rel))
                os.makedirs(os.path.dirname(dest), exist_ok=True)
                shutil.copy2(full_src, dest)
        # Sync run final.jsons
        for run_sub, r_name in [("runs/baseline_fixed/final.json", "A"), ("runs/rev_fixed/final.json", "B"), ("runs/rev_max/final.json", "C")]:
            full_src = os.path.join(drive_root, run_sub)
            if os.path.exists(full_src):
                dest_dir = os.path.join("results/runs", r_name)
                os.makedirs(dest_dir, exist_ok=True)
                shutil.copy2(full_src, os.path.join(dest_dir, "final.json"))

from scripts.make_report import main as make_report_main
make_report_main()


In [ ]:
# Run sample generations for finding F10
from scripts.generate import FIXED_PROMPTS, generate_greedy
from revllm.config import ModelConfig
from revllm.model import TransformerLM
from tokenizers import Tokenizer
import torch

if os.path.exists("data/tokenizer.json"):
    tok = Tokenizer.from_file("data/tokenizer.json")
    print("Tokenizer loaded. Ready for generation analysis.")
print("Analysis phase completed.")
